[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/tensor-forge/blob/main/fable_folder/notebooks/09_attention_build_a_gpt.ipynb)

# ⚒️ Act II · Quest 09 — Attention — Build a GPT

*The architecture that ate the world, from a single dot product to a working mini-GPT.*

⬅️ [08_memory_sequences](08_memory_sequences.ipynb)  •  [10_debugging_dojo](10_debugging_dojo.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## The problem with memory

Your GRU squeezed *everything it read* into one fixed-size hidden vector. Ask it about the first
word of a long paragraph and the memory has faded. **Attention** removes the bottleneck: every
token gets to *look directly at every other token* and take what it needs.

Each token emits three vectors from learned projections:
- **Q**uery — "what am I looking for?"
- **K**ey — "what do I contain?"
- **V**alue — "what do I hand over if you attend to me?"

\[ \text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V \]

### First, feel it with actual numbers

In [ ]:
# 3 tokens, 4 dims. Token 0's query happens to match token 2's key.
Q = torch.tensor([[1., 0, 0, 0], [0., 1, 0, 0], [0., 0, 1, 0]])
K = torch.tensor([[0., 1, 0, 0], [0., 0, 1, 0], [1., 0, 0, 0]])
V = torch.tensor([[10., 0, 0, 0], [0., 20, 0, 0], [0., 0, 30, 0]])

scores = Q @ K.T / math.sqrt(4)
weights = F.softmax(scores, dim=-1)
out = weights @ V

print("attention weights (rows sum to 1):\n", weights.round(decimals=2))
print("\ntoken 0's output:", out[0].round(decimals=1), " <- mostly token 2's value (its key matched!)")

### The causal mask — no peeking at the future

A language model predicts token *t+1* from tokens *≤ t*. Zero out (well, `-inf` out) the upper
triangle of the score matrix and each position can only attend backwards.

In [ ]:
T = 6
scores = torch.randn(T, T)
mask = torch.tril(torch.ones(T, T))
weights = F.softmax(scores.masked_fill(mask == 0, float("-inf")), dim=-1)

plt.figure(figsize=(4, 3.5))
plt.imshow(weights, cmap="viridis"); plt.colorbar()
plt.title("causal attention: strictly lower-triangular"); plt.xlabel("attends to"); plt.ylabel("token")
plt.show()

### Multi-head attention — several conversations at once

In [ ]:
class MultiHeadAttention(nn.Module):
    """h parallel attention heads, each looking for something different."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # split channels into heads: (B, T, C) -> (B, h, T, dk)
        q, k, v = (z.reshape(B, T, self.h, self.dk).transpose(1, 2) for z in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
        causal = torch.tril(torch.ones(T, T, device=x.device))
        att = F.softmax(att.masked_fill(causal == 0, float("-inf")), dim=-1)
        z = (att @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(z)

print("shape check:", MultiHeadAttention(64, 4)(torch.randn(2, 10, 64)).shape)

### The Transformer block: attend, then think

`x + attention(norm(x))` then `x + mlp(norm(x))`. The `x +` **residual connections** are what
let 100-layer stacks train at all — gradients always have a highway back.

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))    # communicate
        x = x + self.mlp(self.norm2(x))     # compute
        return x

class MiniGPT(nn.Module):
    def __init__(self, V, d_model=96, n_heads=4, n_layers=3, block_size=48):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(V, d_model)
        self.pos = nn.Embedding(block_size, d_model)   # attention is order-blind; this restores order
        self.blocks = nn.Sequential(*[Block(d_model, n_heads) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V)

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.norm(self.blocks(x)))

    @torch.no_grad()
    def write(self, idx, n=200, temperature=0.8, top_k=8):
        for _ in range(n):
            logits = self(idx[:, -self.block_size:])[:, -1] / temperature
            if top_k:
                kth = torch.topk(logits, top_k).values[:, -1:]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
        return idx

### Train it

In [ ]:
corpus = (
    "in a hole in the ground there lived a hobbit not a nasty dirty wet hole "
    "filled with the ends of worms and an oozy smell nor yet a dry bare sandy "
    "hole with nothing in it to sit down on or to eat it was a hobbit hole and "
    "that means comfort "
) * 40

chars = sorted(set(corpus))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in corpus], device=device)
V = len(chars)

BS, BLOCK = 48, 48
def batch():
    ix = torch.randint(0, len(data) - BLOCK - 1, (BS,))
    return (torch.stack([data[i:i + BLOCK] for i in ix]),
            torch.stack([data[i + 1:i + BLOCK + 1] for i in ix]))

gpt = MiniGPT(V, block_size=BLOCK).to(device)
opt = torch.optim.AdamW(gpt.parameters(), lr=3e-3)
print("parameters:", sum(p.numel() for p in gpt.parameters()))

STEPS = 500    # 🔼 on a Colab GPU: 3000+ and a real corpus for dramatically better text
for step in range(STEPS):
    xs, ys = batch()
    loss = F.cross_entropy(gpt(xs).reshape(-1, V), ys.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 125 == 0:
        print(f"step {step:4d}: loss {loss.item():.3f}")

In [ ]:
seed = torch.tensor([[stoi["i"]]], device=device)
print("".join(itos[i] for i in gpt.write(seed, n=250)[0].tolist()))

🎉 **That is a GPT.** Token embeddings + positional embeddings + a stack of
attention/MLP blocks + a next-token head. GPT-4 differs in scale (thousands of times more
parameters), tokenizer (subwords, not chars), data (the internet), and engineering — but you
just built the architecture, and you understand every line of it.

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda s: s.strip().lower() in ("v", "value", "values"),
    "attention output is a weighted sum of the ___ vectors (one letter)")
_register("attn_fn", 20,
    lambda f: (lambda o: torch.is_tensor(o) and o.shape == (2, 5, 8)
               and torch.allclose(f(torch.ones(1, 3, 4), torch.ones(1, 3, 4), torch.arange(12.).reshape(1, 3, 4)),
                                   torch.arange(12.).reshape(1, 3, 4).mean(1, keepdim=True).expand(1, 3, 4), atol=1e-5))(
        f(torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8))),
    "def attn(q, k, v): w = softmax(q @ k.transpose(-2,-1) / sqrt(d_k), dim=-1); return w @ v   (no mask)")
_register("causal_check", 15,
    lambda w: torch.is_tensor(w) and w.shape[0] == w.shape[1] and torch.allclose(w.triu(1), torch.zeros_like(w), atol=1e-6)
              and torch.allclose(w.sum(-1), torch.ones(w.shape[0]), atol=1e-5),
    "build a causal attention weight matrix for T=7 from random scores; upper triangle must be 0, rows sum to 1")
_register("gpt_low", 15,
    lambda l: l < 0.9,
    "train the MiniGPT until loss < 0.9 (more steps / slightly bigger model); submit loss.item()")

In [ ]:
check("warmup", "V")

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `attn_fn` (20 XP) — implement plain (unmasked) scaled dot-product attention `attn(q, k, v)`; submit the function.
- `causal_check` (15 XP) — build a valid `7×7` causal attention weight matrix from random scores; submit it.
- `gpt_low` (15 XP) — push MiniGPT training loss under `0.9`; submit the loss value.

In [ ]:
# ⚔️ your attempts here...

# xp_report()